In [23]:
#Instalación de dependencias
!pip install -q -U langchain langchain-core langchain-community langchain-groq langchain-huggingface sentence-transformers faiss-cpu

In [24]:
import os
import json
from datetime import datetime
from getpass import getpass

# Forzamos pedir la API Key de Groq (empieza con gsk_...)
os.environ["GROQ_API_KEY"] = getpass("Ingresa tu GROQ_API_KEY (de console.groq.com): ")

# 1. SIMULACIÓN DE BASE DE DATOS DE SENSORES IoT (Para Herramienta de Consulta)
SENSORES_DB = {
    "CAM-01": {
        "nombre": "Cámara de Congelados Principal",
        "temp_actual_c": -5.2,
        "temp_objetivo_c": -18.0,
        "presion_compresor_psi": 185,
        "humedad_pct": 88,
        "estado_puerta": "CERRADA",
        "alerta_activa": "ALTA_TEMPERATURA_Y_SOBREPRESION"
    },
    "CAM-02": {
        "nombre": "Cámara de Mantención Lácteos",
        "temp_actual_c": 2.8,
        "temp_objetivo_c": 3.0,
        "presion_compresor_psi": 68,
        "humedad_pct": 65,
        "estado_puerta": "CERRADA",
        "alerta_activa": "NINGUNA"
    },
    "CAM-03": {
        "nombre": "Túnel de Enfriamiento Rápido",
        "temp_actual_c": -8.0,
        "temp_objetivo_c": -22.0,
        "presion_compresor_psi": 42,
        "humedad_pct": 94,
        "estado_puerta": "ABIERTA_PROLONGADA",
        "alerta_activa": "BLOQUEO_HIELO_EVAPORADOR"
    }
}

# Archivo JSON para guardar tickets generados (Herramienta de Escritura)
TICKETS_FILE = "ordenes_trabajo.json"
with open(TICKETS_FILE, "w", encoding="utf-8") as f:
    json.dump([], f)

print("✅ Entorno y base de datos IoT inicializados.")

Ingresa tu GROQ_API_KEY (de console.groq.com): ··········
✅ Entorno y base de datos IoT inicializados.


In [25]:
import warnings
warnings.filterwarnings("ignore")

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Documentos técnicos y protocolos de la empresa (Base de Conocimiento Semántica)
documentos_manuales = [
    Document(
        page_content="PROTOCOLO FALLA F-101 (Alta Temperatura y Sobrepresión > 150 PSI): Si una cámara presenta temperatura sobre -10°C y presión del compresor mayor a 150 PSI, indica obstrucción en el condensador o falla del ventilador de tiro forzado. Acción inmediata: 1) Purgar línea de alta presión, 2) Limpiar serpentín condensador, 3) Generar orden de trabajo con prioridad CRITICA.",
        metadata={"fuente": "Manual_Mantenimiento_FrigoTech_v2.pdf", "codigo": "F-101"}
    ),
    Document(
        page_content="PROTOCOLO FALLA F-204 (Bloqueo de Hielo en Evaporador y Humedad > 90%): Ocurre por apertura prolongada de puertas y baja presión de succión (< 50 PSI). Acción inmediata: 1) Cerrar accesos de inmediato, 2) Activar ciclo de deshielo por resistencias eléctricas (25 minutos), 3) Generar orden de trabajo con prioridad ALTA.",
        metadata={"fuente": "Manual_Mantenimiento_FrigoTech_v2.pdf", "codigo": "F-204"}
    ),
    Document(
        page_content="REGLA DE PRIORIZACIÓN OPERATIVA (SLA): Prioridad CRITICA requiere intervención en menos de 30 minutos (riesgo de pérdida de cadena de frío en congelados). Prioridad ALTA requiere atención en menos de 2 horas. Prioridad PREVENTIVA se agenda en ventana de 48 horas.",
        metadata={"fuente": "Politica_SLA_Operaciones.pdf", "codigo": "SLA-01"}
    ),
    Document(
        page_content="HISTORIAL DE MANTENCIÓN RECIENTE: La cámara CAM-01 tuvo cambio de válvula de expansión hace 15 días. La cámara CAM-03 reportó falla en el sensor magnético de puerta la semana pasada.",
        metadata={"fuente": "Bitacora_Historica_2026.pdf", "codigo": "HIST-01"}
    )
]

# Embeddings locales multilingües de HuggingFace (corren directo en Colab sin API Key)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(documentos_manuales, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ Memoria semántica (FAISS + HuggingFace Local) indexada sin errores.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Memoria semántica (FAISS + HuggingFace Local) indexada sin errores.


In [26]:
from langchain_core.tools import tool

@tool
def consultar_telemetria_camara(camara_id: str) -> str:
    """HERRAMIENTA DE CONSULTA: Obtiene en tiempo real las lecturas de sensores IoT (temperatura, presión, humedad) de una cámara frigorífica (ej. CAM-01, CAM-02, CAM-03) o de 'TODAS'."""
    camara_id = camara_id.upper().strip()
    if camara_id == "TODAS":
        return json.dumps(SENSORES_DB, indent=2, ensure_ascii=False)
    datos = SENSORES_DB.get(camara_id)
    if not datos:
        return f"Error: No se encontró la cámara {camara_id}. Cámaras válidas: CAM-01, CAM-02, CAM-03."
    return json.dumps({camara_id: datos}, indent=2, ensure_ascii=False)

@tool
def buscar_protocolo_tecnico(consulta_sintomas: str) -> str:
    """HERRAMIENTA DE RECUPERACIÓN SEMÁNTICA (IE4): Busca en los manuales técnicos, SLA e historial de la empresa soluciones y protocolos según los síntomas o alertas detectadas."""
    docs = retriever.invoke(consulta_sintomas)
    resultado = []
    for d in docs:
        resultado.append(f"[{d.metadata['fuente']} - {d.metadata['codigo']}]: {d.page_content}")
    return "\n\n".join(resultado)

@tool
def calcular_criticidad_y_riesgo(temp_actual: float, temp_objetivo: float, presion_psi: float) -> str:
    """HERRAMIENTA DE RAZONAMIENTO Y CÁLCULO: Evalúa matemáticamente la desviación térmica y de presión para determinar el nivel de riesgo operativo y prioridad sugerida."""
    desviacion_temp = float(temp_actual) - float(temp_objetivo)
    riesgo = "NORMAL"
    prioridad_sugerida = "PREVENTIVA"

    if desviacion_temp >= 10 or float(presion_psi) >= 150:
        riesgo = "PELIGRO DE RUPTURA DE CADENA DE FRÍO / FALLA MECÁNICA"
        prioridad_sugerida = "CRITICA"
    elif desviacion_temp >= 5 or float(presion_psi) < 50:
        riesgo = "DESVIACIÓN MODERADA / INEFICIENCIA TÉRMICA"
        prioridad_sugerida = "ALTA"

    return json.dumps({
        "desviacion_termica_grados": round(desviacion_temp, 2),
        "presion_evaluada_psi": presion_psi,
        "diagnostico_riesgo": riesgo,
        "prioridad_recomendada": prioridad_sugerida
    }, ensure_ascii=False)

@tool
def registrar_orden_trabajo(camara_id: str, prioridad: str, diagnostico: str, plan_accion: str) -> str:
    """HERRAMIENTA DE ESCRITURA: Registra y guarda una Orden de Trabajo (Ticket de Mantenimiento) en el sistema persistente JSON de la empresa."""
    with open(TICKETS_FILE, "r", encoding="utf-8") as f:
        tickets = json.load(f)

    nuevo_ticket = {
        "ticket_id": f"OT-2026-{len(tickets) + 1:03d}",
        "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "camara_id": camara_id.upper(),
        "prioridad": prioridad.upper(),
        "diagnostico": diagnostico,
        "plan_accion": plan_accion,
        "estado": "ASIGNADO_A_TERRENO"
    }
    tickets.append(nuevo_ticket)

    with open(TICKETS_FILE, "w", encoding="utf-8") as f:
        json.dump(tickets, f, indent=2, ensure_ascii=False)

    return f"✅ ÉXITO: Orden de trabajo {nuevo_ticket['ticket_id']} creada y almacenada con prioridad {prioridad.upper()} para {camara_id}."

tools = [consultar_telemetria_camara, buscar_protocolo_tecnico, calcular_criticidad_y_riesgo, registrar_orden_trabajo]
print("✅ Herramientas de Consulta, RAG, Razonamiento y Escritura configuradas.")

✅ Herramientas de Consulta, RAG, Razonamiento y Escritura configuradas.


In [27]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# Motor LLM ultrarrápido Llama 3.3 70B vía Groq conectado a las 4 herramientas
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
llm_con_herramientas = llm.bind_tools(tools)
mapa_herramientas = {t.name: t for t in tools}

system_prompt = """Eres 'FrigoAgent', un Agente Autónomo de Ingeniería de Mantenimiento para sistemas de refrigeración industrial.
Tu objetivo es supervisar sensores IoT, diagnosticar fallas con respaldo en manuales técnicos y gestionar órdenes de trabajo.

ESQUEMA DE PLANIFICACIÓN DE TAREAS OBLIGATORIO :
Cuando el usuario solicite revisar una cámara o reportar un problema, DEBES seguir esta secuencia lógica:
1. PASO 1 (Consulta): Usa `consultar_telemetria_camara` para obtener los datos reales de los sensores.
2. PASO 2 (Razonamiento): Si detectas anomalías, usa `calcular_criticidad_y_riesgo` pasando los valores numéricos de temperatura y presión.
3. PASO 3 (Recuperación Semántica): Usa `buscar_protocolo_tecnico` para consultar en los manuales el código de falla, causas y pasos de reparación.
4. PASO 4 (Toma de Decisiones y Escritura):
   - Si el estado es NORMAL (ej. CAM-02), informa que los parámetros están estables y NO generes orden de trabajo.
   - Si la prioridad es ALTA o CRITICA, genera automáticamente la orden de trabajo usando `registrar_orden_trabajo` detallando el plan de acción del manual.

Responde siempre en español de forma estructurada, técnica y profesional, detallando el plan que ejecutaste."""

# Diccionario de Memoria de Corto Plazo por Sesión
store_sesiones = {}

class OrquestadorFrigoAgent:
    def invoke(self, inputs: dict, config: dict = None):
        session_id = config.get("configurable", {}).get("session_id", "default") if config else "default"
        if session_id not in store_sesiones:
            store_sesiones[session_id] = [SystemMessage(content=system_prompt)]

        historial = store_sesiones[session_id]
        historial.append(HumanMessage(content=inputs["input"]))

        # Ciclo autónomo ReAct / Tool-Calling (Planificación y Ejecución)
        for paso in range(8):
            respuesta = llm_con_herramientas.invoke(historial)
            historial.append(respuesta)

            # Si el agente ya terminó de usar herramientas, entrega la respuesta final
            if not respuesta.tool_calls:
                return {"output": respuesta.content}

            # Ejecución autónoma de las herramientas invocadas por el agente
            for tool_call in respuesta.tool_calls:
                nombre_fn = tool_call["name"]
                args_fn = tool_call["args"]
                print(f"⚙️ [Ejecutando Herramienta]: {nombre_fn} | Args: {args_fn}")

                resultado_fn = mapa_herramientas[nombre_fn].invoke(args_fn)
                print(f"   ↳ Resultado: {str(resultado_fn)[:160]}...\n")

                historial.append(
                    ToolMessage(content=str(resultado_fn), tool_call_id=tool_call["id"])
                )

        return {"output": historial[-1].content}

agente_con_memoria = OrquestadorFrigoAgent()
print("✅ Agente FrigoAgent compilado y listo con Memoria de Sesión y Tool-Calling.")

✅ Agente FrigoAgent compilado y listo con Memoria de Sesión y Tool-Calling.


In [31]:
import time
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage

# 1. Configuramos el modelo con max_tokens=600 para NUNCA superar el límite de 1000 OTPM de Groq
# Usamos openai/gpt-oss-120b o qwen/qwen3.8-27b que están activos en tu cuenta
modelo_activo = "openai/gpt-oss-120b"  # Si prefieres Qwen, cámbialo a "qwen/qwen3.8-27b"

llm = ChatGroq(
    model=modelo_activo,
    temperature=0.1,
    max_tokens=600,       # <-- EVITA EL ERROR 429 DE OUTPUT TOKENS
    max_retries=3
)
llm_con_herramientas = llm.bind_tools(tools)
print(f"✅ Modelo configurado con control de cuota (max_tokens=600): {modelo_activo}\n")

# 2. Reiniciamos la memoria de esta sesión y el archivo de tickets para una corrida limpia
store_sesiones["turno_planta_01"] = [SystemMessage(content=system_prompt)]
with open(TICKETS_FILE, "w", encoding="utf-8") as f:
    json.dump([], f)

config_sesion = {"configurable": {"session_id": "turno_planta_01"}}

print("="*80)
print("PRUEBA 1: Revisión general y priorización automática ante múltiples cámaras")
print("="*80)
respuesta_1 = agente_con_memoria.invoke(
    {"input": "Hola FrigoAgent. Haz un barrido de TODAS las cámaras de la planta, evalúa cuáles están en peligro y ejecuta los protocolos y órdenes de trabajo solo en la más crítica primero. Sé conciso."},
    config=config_sesion
)
print("\n🤖 RESPUESTA DEL AGENTE:\n", respuesta_1["output"])

# Pausa breve para refrescar la ventana de tokens por minuto (TPM/OTPM) de la API gratuita
print("\n⏳ Esperando 15 segundos para refrescar cuota de tokens por minuto de Groq...")
time.sleep(15)

print("\n" + "="*80)
print("PRUEBA 2: Continuidad de Memoria (IE3) y atención de la segunda alerta")
print("="*80)
respuesta_2 = agente_con_memoria.invoke(
    {"input": "Perfecto. Ahora procesa la otra cámara que tenía problemas pendientes en el barrido anterior y déjale su ticket creado. Sé conciso."},
    config=config_sesion
)
print("\n🤖 RESPUESTA DEL AGENTE:\n", respuesta_2["output"])

print("\n" + "="*80)
print("VERIFICACIÓN DE ESCRITURA PERSISTENTE (Archivo ordenes_trabajo.json):")
print("="*80)
with open(TICKETS_FILE, "r", encoding="utf-8") as f:
    print(f.read())

✅ Modelo configurado con control de cuota (max_tokens=600): openai/gpt-oss-120b

PRUEBA 1: Revisión general y priorización automática ante múltiples cámaras
⚙️ [Ejecutando Herramienta]: consultar_telemetria_camara | Args: {'camara_id': 'CAM-01'}
   ↳ Resultado: {
  "CAM-01": {
    "nombre": "Cámara de Congelados Principal",
    "temp_actual_c": -5.2,
    "temp_objetivo_c": -18.0,
    "presion_compresor_psi": 185,
    "...

⚙️ [Ejecutando Herramienta]: consultar_telemetria_camara | Args: {'camara_id': 'CAM-02'}
   ↳ Resultado: {
  "CAM-02": {
    "nombre": "Cámara de Mantención Lácteos",
    "temp_actual_c": 2.8,
    "temp_objetivo_c": 3.0,
    "presion_compresor_psi": 68,
    "humeda...

⚙️ [Ejecutando Herramienta]: consultar_telemetria_camara | Args: {'camara_id': 'CAM-03'}
   ↳ Resultado: {
  "CAM-03": {
    "nombre": "Túnel de Enfriamiento Rápido",
    "temp_actual_c": -8.0,
    "temp_objetivo_c": -22.0,
    "presion_compresor_psi": 42,
    "hum...

⚙️ [Ejecutando Herramienta]: calc